Our goal is to build an assistant that can answer questions by searching the web right now. We won’t be using any paid APIs from OpenAI or Google. This is all you.

Here’s the toolkit we will be using for this task:

Ollama: A fantastic tool that lets you download and run powerful open-source LLMs (like Meta’s Llama 3 or Mistral’s Mistral) right on your computer.

LangChain: The core framework we’ll use to build our application’s chain of logic.

DuckDuckGo Search: A free Python library that lets us perform web searches without needing an API key.

Let’s get started!

Step 1: Set Up Your Environment

First, you need to install Ollama. Go to ollama.com and download the app for your OS (Mac, Windows, or Linux).

Once installed, open your terminal and pull a model. Let’s use Llama 3 8B, a powerful and fast model:

In [1]:
!ollama pull llama3:8b

/bin/bash: line 1: ollama: command not found


In [2]:
# Install zstd dependency
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
print("Starting Ollama server...")
get_ipython().system_raw('ollama serve &')
print("Ollama server started in the background. Please wait a few seconds for it to initialize.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,996 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,917 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcont

In [3]:
pip install langchain langchain_community langchain_ollama duckduckgo-search ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


Step 2: Assemble Your Components in Python

Create a new Python file (assistant.py). Let’s import our tools and set up the main components:

In [4]:
from langchain_ollama import OllamaLLM
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

Now, let’s initialise the two main parts:

The LLM: Llama 3, running via Ollama.
The Search Tool: DuckDuckGo.

In [5]:
# We specify the model we pulled in Step 1
llm = OllamaLLM(model="llama3:8b")

# This tool will run a web search
search = DuckDuckGoSearchRun()

Step 3: Define the Chain with LCEL

This is the key part of our real-time AI Assistant. We need to tell LangChain how to route the information. We’ll use the LangChain Expression Language (LCEL), which looks like Python pipes.

First, we create a prompt template. This is the briefing we give to our LLM. Notice how we have placeholders for {context} (the search results) and {question}:

In [6]:
# This is the prompt template, our instruction manual for the LLM
prompt = ChatPromptTemplate.from_template(
    """You are a helpful AI assistant. You must answer the user's question
    based *only* on the following search results. If the search results
    are empty or do not contain the answer, say 'I could not find
    any information on that.'

    Search Results:
    {context}

    Question:
    {question}
    """
)

Now, we build the chain itself. Read the comments in the code to see how the data flows:

In [7]:
# This is our RAG chain
chain = (
    RunnablePassthrough.assign(
        # "context" is a new key we add to the dictionary.
        # Its value is the *output* of running the 'search' tool
        # with the original 'question' as input.
        context=lambda x: search.run(x["question"])
    )
    | prompt  # The dictionary (now with 'context' and 'question') is "piped" into the prompt
    | llm     # The formatted prompt is "piped" into the LLM
)

That RunnablePassthrough.assign is the key. It takes the original input (a dictionary with a question key), retains it, and assigns a new key called ‘context’ by running the search tool.

Step 4: Run Your Real-Time Assistant!

That’s it. The chain is built. Now, all we have to do is invoke it:

In [ ]:
print("🤖 Hello! I'm a real-time AI assistant. What's new?")
while True:
    try:
        user_query = input("You: ")
        if user_query.lower() in ["exit", "quit"]:
            print("🤖 Goodbye!")
            break

        print("🤖 Thinking...")

        # This one line runs the whole RAG process
        response = chain.invoke({"question": user_query})

        print(f"🤖: {response}")

    except Exception as e:
        print(f"An error occurred: {e}")

🤖 Hello! I'm a real-time AI assistant. What's new?
You: hello
🤖 Thinking...
An error occurred: model 'llama3:8b' not found (status code: 404)


In [ ]:
!ollama list

Run your Python file: python assistant.py.

Final Words

RAG isn’t just a technical trick to get around knowledge cut-offs. It’s a profound step toward grounded AI. It connects the abstract, statistical intelligence of an LLM to the concrete, verifiable facts of the real world.

By building this, you’ve done more than pipe some data. You’ve given your AI a sense of now, a library and the curiosity to use it. You’ve built an assistant that doesn’t just know things; it’s ready to learn things.